In [267]:
# Giải thích: Thêm chú thích phù hợp ở đây
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ***Làm sạch dữ liệu***

In [268]:
# Xác định thư mục gốc của dự án (Analyze&ForecastSaleSystem)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Tạo đường dẫn an toàn đến file
DATA_PATH = PROJECT_ROOT / 'data' / 'amazon_products_sales_data_uncleaned.csv'

# Import dataset
df = pd.read_csv(DATA_PATH)

df.head()


,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6 out of 5 stars,"35,882",10K+ bought in past month,NaN,basic variant price: $162.24,No Discount,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,/Apple-Cancellation-Transparency-Personalized-...,2025-08-21 11:14:29
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8 out of 5 stars,"28,988",10K+ bought in past month,NaN,basic variant price: $72.74,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...,2025-08-21 11:14:29


In [269]:
# Check the size of the DataFrame (rows x columns)
print(f"The DataFrame has {df.shape[0]} rows and {df.shape[1]} columns.")

# Display all column names in the DataFrame
print(f"The DataFrame columns are: {', '.join(df.columns)}")


The DataFrame has 42675 rows and 16 columns.
The DataFrame columns are: title, rating, number_of_reviews, bought_in_last_month, current/discounted_price, price_on_variant, listed_price, is_best_seller, is_sponsored, is_couponed, buy_box_availability, delivery_details, sustainability_badges, image_url, product_url, collected_at


In [270]:
# Create a copy of the original DataFrame to work with
df1 = df.copy()

# Check for duplicate rows in the copied DataFrame
num_duplicates = df1.duplicated().sum()
print(f"Number of duplicate rows in df1: {num_duplicates}")


Number of duplicate rows in df1: 0


<h1 style="background:#374151;color:#f9fafb;padding:10px 14px;border-radius:6px;display:inline-block;font-size:28px;font-family:'Segoe UI',sans-serif;margin:0;">
  Các vấn đề với dữ liệu
</h1>


<br>
<h2 style="background:#4b5563;color:#f9fafb;padding:8px 12px;border-radius:6px;display:inline-block;font-size:22px;font-family:'Segoe UI',sans-serif;margin:10px 0;">
  a). Dữ liệu lộn xộn
</h2>
<br>


- **rating**: Cột chứa văn bản không cần thiết như "out of five stars" và kiểu dữ liệu không chính xác (`object`).

- **current/discounted_price**: Cột có nhiều vấn đề. Nhiều giá trị bị thiếu vì một số giá nằm dưới cột **price_on_variant**.

- **price_on_variant**:  
  - Cột chứa từ ngữ không cần thiết, nhiều giá trị có thông tin không chính xác thay vì "giá" vì trong quá trình cạo dữ liệu, nơi các sản phẩm không thể dựa vào các biến thể, giá bị thiếu và vì lý do này, các thông tin khác được lấy một cách không cẩn thận.
  
  - Cột cũng chứa các ký tự đặc biệt như `$` và `,`.

- **listed_price**:  
  - Cột chứa văn bản "No Discount" ở những hàng không có giảm giá thay vì giá. Chúng ta có thể thay thế các giá trị này bằng **current/discounted_price** khi giá niêm yết bị thiếu.
  - Cũng chứa các ký hiệu `$` và `S`.

- **is_couponed** và **buy_box_availability**: Các cột cần một số cải tiến và tiêu chuẩn hóa.

- **delivery_details**: Cột có nhiều vấn đề; định dạng không nhất quán là vấn đề chính và nó cũng có kiểu dữ liệu sai.

- **collected_at**: Cột có kiểu dữ liệu sai và cần được sửa chữa.

- **number_of_reviews**: Cột có dấu phẩy (ví dụ: `2,563`) cần được loại bỏ để chuyển đổi sang số.

- Xử lý các giá trị thiếu trong **current/discounted_price**:  
  - Có nhiều giá trị NaN, điền chúng sau bằng các giá trị **price_on_variant** khi bị thiếu.  
  - **listed_price**: Điền các giá trị thiếu bằng **current/discounted_price** khi không có giảm giá.  
  - **delivery_details**: Chuyển đổi thành datetime.

- **product_url**: Cột có vấn đề về tính đầy đủ.

<br>
<h2 style="background:#4b5563;color:#f9fafb;padding:8px 12px;border-radius:6px;display:inline-block;font-size:22px;font-family:'Segoe UI',sans-serif;margin:10px 0;">
  b). Dữ liệu không sạch
</h2>
<br>


- **product_url**: Cần hoàn thiện với tên miền web chính.

- **category**: Cột bị thiếu trong tập dữ liệu, đây là một vấn đề lớn. Chúng ta có thể tạo một cột mới có tên **category** và gán danh mục cho từng sản phẩm bằng cách sử dụng cột **title**.

<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Quy trình làm sạch cột 'current/discounted_price'
</h3>

In [271]:
# fills missing values in current orice col with basic varient price

df1['current/discounted_price'].isna().sum()

np.int64(11749)

In [272]:
# Extract the actual price from 'price_on_variant' column by splitting on ':' and taking the second part
df1['price_on_variant'] = df1['price_on_variant'].str.split(":").str.get(1)


In [273]:
# Set 'price_on_variant' to NaN for rows that do not contain the '$' sign
df1.loc[~df1['price_on_variant'].str.contains(r'\$', na=False), 'price_on_variant'] = np.nan

In [274]:
# Clean 'price_on_variant' by stripping extra spaces and taking the first part if multiple values exist
df1['price_on_variant'] = df1['price_on_variant'].str.strip().str.split(" ").str.get(0)

In [275]:
# Fill missing values in 'current/discounted_price' with the corresponding 'price_on_variant' values
df1['current/discounted_price'] = df1['current/discounted_price'].fillna(df1['price_on_variant'])

In [276]:
# Clean 'current/discounted_price' by removing '$' and ',' then convert to float
df1['current/discounted_price'] = df1['current/discounted_price'].str.replace(r"\$", "", regex=True).str.replace(r",", "").astype(float)

In [277]:
print(f"Remaining missing values in 'current/discounted_price': {df1['current/discounted_price'].isna().sum()}")


Remaining missing values in 'current/discounted_price': 2062


In [278]:
# Display a random sample of 2 rows from the DataFrame to inspect the data
df1.sample(2)

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
33641,"Canon CLI-251XL Genuine Yellow Ink Tank, Compa...",4.8 out of 5 stars,945,400+ bought in past month,21.99,NaN,$23.99,No Badge,Organic,No Coupon,Add to cart,"FREE delivery Wed, Sep 3 on $35 of items shipp...",NaN,https://m.media-amazon.com/images/I/71FKqmlDSL...,/Canon-CLI-251XL-Yellow-Compatible-MG6320/dp/B...,2025-08-29 11:34:22
34797,"Pyle Dash Cam, Rearview Mirror Monitor, Night ...",3.8 out of 5 stars,30,NaN,139.99,NaN,No Discount,No Badge,Sponsored,No Coupon,Add to cart,"FREE delivery Mon, Sep 8",NaN,https://m.media-amazon.com/images/I/61NIPEjnc-...,/sspa/click?ie=UTF8&spc=MToxOTI3ODkyMzg2NzA2Mz...,2025-08-30 00:06:24


<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Quy trình làm sạch cột 'rating'
</h3>

In [279]:
# Display the 'rating' column
df1['rating']

0        4.6 out of 5 stars
1        4.3 out of 5 stars
2        4.6 out of 5 stars
3        4.6 out of 5 stars
4        4.8 out of 5 stars
                ...        
42670    5.0 out of 5 stars
42671    4.2 out of 5 stars
42672    4.3 out of 5 stars
42673    4.7 out of 5 stars
42674    4.4 out of 5 stars
Name: rating, Length: 42675, dtype: str

In [280]:
# Count the number of missing values in the 'rating' column
num_missing_ratings = df1['rating'].isna().sum()
print(f"Number of missing values in 'rating': {num_missing_ratings}")


Number of missing values in 'rating': 1024


In [281]:
# Clean the 'rating' column by removing "out of 5 stars", stripping extra spaces, and converting to float
df1['rating'] = df1['rating'].str.replace(r"out of 5 stars", "").str.strip().astype(float)

In [282]:
# Display the count of each unique value in the 'rating' column, including NaNs
rating_counts = df1['rating'].value_counts(dropna=False)
print(rating_counts)


rating
4.6    6151
4.4    5525
4.5    5359
4.7    4664
4.8    4230
4.3    2927
4.2    2837
4.1    1959
4.0    1465
3.9    1316
3.8    1083
NaN    1024
5.0     995
4.9     704
3.6     666
3.7     604
3.2     363
3.5     242
3.4     216
3.0     148
2.0     143
2.7      15
1.5      15
3.3       6
2.8       4
1.0       4
2.4       3
2.9       2
3.1       2
2.3       1
2.5       1
2.6       1
Name: count, dtype: int64


In [283]:
# Display a concise summary of the DataFrame, including column types, non-null counts, and memory usage
df1.info()


<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   title                     42675 non-null  str    
 1   rating                    41651 non-null  float64
 2   number_of_reviews         41651 non-null  str    
 3   bought_in_last_month      39458 non-null  str    
 4   current/discounted_price  40613 non-null  float64
 5   price_on_variant          20071 non-null  object 
 6   listed_price              42675 non-null  str    
 7   is_best_seller            42675 non-null  str    
 8   is_sponsored              42675 non-null  str    
 9   is_couponed               42675 non-null  str    
 10  buy_box_availability      28022 non-null  str    
 11  delivery_details          30955 non-null  str    
 12  sustainability_badges     3408 non-null   str    
 13  image_url                 42675 non-null  str    
 14  product_url      

<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Quy trình làm sạch cột 'number_of_reviews'
</h3>

In [284]:
df1['number_of_reviews']

0           375
1         2,457
2         3,044
3        35,882
4        28,988
          ...  
42670         1
42671        20
42672        57
42673     7,102
42674        75
Name: number_of_reviews, Length: 42675, dtype: str

In [285]:
# Clean 'number_of_reviews' by removing commas, stripping spaces, and converting to float
df1['number_of_reviews'] = df1['number_of_reviews'].str.replace(",", "").str.strip().astype(float)

In [286]:
# Count the number of missing values in 'number_of_reviews' column
num_missing_reviews = df1['number_of_reviews'].isna().sum()
print(f"Number of missing values in 'number_of_reviews': {num_missing_reviews}")


Number of missing values in 'number_of_reviews': 1024


In [287]:
# Display a concise summary of the DataFrame to check column types, non-null counts, and memory usage
df1.info()


<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   title                     42675 non-null  str    
 1   rating                    41651 non-null  float64
 2   number_of_reviews         41651 non-null  float64
 3   bought_in_last_month      39458 non-null  str    
 4   current/discounted_price  40613 non-null  float64
 5   price_on_variant          20071 non-null  object 
 6   listed_price              42675 non-null  str    
 7   is_best_seller            42675 non-null  str    
 8   is_sponsored              42675 non-null  str    
 9   is_couponed               42675 non-null  str    
 10  buy_box_availability      28022 non-null  str    
 11  delivery_details          30955 non-null  str    
 12  sustainability_badges     3408 non-null   str    
 13  image_url                 42675 non-null  str    
 14  product_url      

<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Quy trình làm sạch cột 'bought_in_last_month'
</h3>

In [288]:
df1['bought_in_last_month']

0        300+ bought in past month
1         6K+ bought in past month
2         2K+ bought in past month
3        10K+ bought in past month
4        10K+ bought in past month
                   ...            
42670    100+ bought in past month
42671    200+ bought in past month
42672     50+ bought in past month
42673    500+ bought in past month
42674     50+ bought in past month
Name: bought_in_last_month, Length: 42675, dtype: str

In [289]:
# Count the number of missing values in the 'bought_in_last_month' column
num_missing_bought = df1['bought_in_last_month'].isna().sum()
print(f"Number of missing values in 'bought_in_last_month': {num_missing_bought}")


Number of missing values in 'bought_in_last_month': 3217


In [290]:
# Clean 'bought_in_last_month' by removing text and converting shorthand 'K' to full numbers
df1['bought_in_last_month'] = df1['bought_in_last_month'].str.replace("+ bought in past month","").str.strip().str.replace("K","000")


In [291]:
# Keep only numeric values in 'bought_in_last_month'; set non-numeric entries to NaN
df1['bought_in_last_month'] = df1['bought_in_last_month'].where(
    df1['bought_in_last_month'].str.isdigit(),
    np.nan
)

In [292]:
# Keep only numeric values in 'bought_in_last_month', set non-numeric entries to NaN, and convert to nullable integer
df1['bought_in_last_month'] = (
    df1['bought_in_last_month']
    .where(df1['bought_in_last_month'].str.isdigit(), np.nan)
    .astype('Int64')  # Use nullable integer type to allow NaNs
)


In [293]:
# Count the number of missing values in the 'bought_in_last_month' column after cleaning
num_missing_bought = df1['bought_in_last_month'].isna().sum()
print(f"Number of missing values in 'bought_in_last_month' after cleaning: {num_missing_bought}")


Number of missing values in 'bought_in_last_month' after cleaning: 10511


In [294]:
# Display the count of each unique value in 'bought_in_last_month' after cleaning
bought_counts = df1['bought_in_last_month'].value_counts()
print(bought_counts)


bought_in_last_month
100       8801
50        5967
200       5645
300       2842
500       2351
1000      2084
400       1436
20000      772
2000       426
800        280
3000       249
10000      229
600        229
4000       196
5000       120
700        102
90000       92
6000        81
900         63
7000        49
9000        44
8000        36
30000       28
40000       14
100000      13
50000        6
70000        4
60000        3
80000        2
Name: count, dtype: Int64


In [295]:
# Display a concise summary of the DataFrame to check column types, non-null counts, and memory usage after all cleaning steps
df1.info()


<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   title                     42675 non-null  str    
 1   rating                    41651 non-null  float64
 2   number_of_reviews         41651 non-null  float64
 3   bought_in_last_month      32164 non-null  Int64  
 4   current/discounted_price  40613 non-null  float64
 5   price_on_variant          20071 non-null  object 
 6   listed_price              42675 non-null  str    
 7   is_best_seller            42675 non-null  str    
 8   is_sponsored              42675 non-null  str    
 9   is_couponed               42675 non-null  str    
 10  buy_box_availability      28022 non-null  str    
 11  delivery_details          30955 non-null  str    
 12  sustainability_badges     3408 non-null   str    
 13  image_url                 42675 non-null  str    
 14  product_url      

In [296]:
df1.sample(5)

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
38478,"Keychron B6 Pro Ultra-Thin Wireless Keyboard, ...",4.0,68.0,200,38.24,NaN,$44.99,No Badge,Organic,No Coupon,Add to cart,"FREE delivery Thu, Sep 4Or fastest delivery To...",NaN,https://m.media-amazon.com/images/I/71o79KEdeu...,/Keychron-B6-Pro-Ultra-Thin-Connection/dp/B0D5...,2025-08-30 00:36:47
40678,"Xerox B235DNI All-In-One Printer, Laser, B&W, ...",3.6,97.0,100,229.99,$176.73,No Discount,No Badge,Organic,No Coupon,Add to cart,FREE delivery Sep 4 - 8,NaN,https://m.media-amazon.com/images/I/41xdAqUpWk...,/Xerox-B235-Multifunction-Printer-Wireless/dp/...,2025-08-30 19:39:47
14385,"Lenovo USI Stylus Pen, Chrome OS Support, 4,09...",3.9,1702.0,200,29.99,NaN,$43.99,No Badge,Organic,No Coupon,Add to cart,"Delivery Tue, Sep 2",NaN,https://m.media-amazon.com/images/I/41q6mz71if...,/Lenovo-Pressure-Sensitivity-Chromebook-GX81B1...,2025-08-24 22:25:59
13348,Insta360 Multi Mount,4.8,84.0,100,40.99,NaN,$45.99,No Badge,Organic,No Coupon,Add to cart,"Delivery Thu, Sep 4",NaN,https://m.media-amazon.com/images/I/51xQoEaonO...,/Insta360-CINSTAV-F-Multi-Mount/dp/B0CNKKY9NH/...,2025-08-24 22:17:52
23692,ASURION 4 Year Home Improvement Protection Pla...,4.4,613.0,<NA>,93.99,$93.99,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/5179luWipK...,/ASURION-Year-Improvement-Protection-600-699-9...,2025-08-27 07:51:40


<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Quy trình làm sạch cột 'listted_price'
</h3>

In [297]:
num_missing_listed = df1['listed_price'].isna().sum()
print(f"Number of missing values in 'listed_price' (Before): {num_missing_listed}")

Number of missing values in 'listed_price' (Before): 0


In [298]:
# Clean 'listed_price' by removing '$' and ',' and stripping extra spaces
df1['listed_price'] = (
    df1['listed_price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

In [299]:
# Replace "No Discount" entries in 'listed_price' with NaN to handle missing values
df1['listed_price'] = df1['listed_price'].replace(
    ['No Discount', 'nan', 'NaN', ''], np.nan
)

In [300]:
# Count the number of missing values in the 'listed_price' column after cleaning
num_missing_listed = df1['listed_price'].isna().sum()
print(f"Number of missing values in 'listed_price': {num_missing_listed}")

Number of missing values in 'listed_price': 30364


In [301]:
# Làm sạch cột current/discounted_price trước
df1['current/discounted_price'] = (
    df1['current/discounted_price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

# Fill missing values in 'listed_price' with corresponding values from 'current/discounted_price'
df1['listed_price'] = df1['listed_price'].fillna(
    df1['current/discounted_price']
)


In [302]:
# Convert 'listed_price' column to float type for numeric operations
df1['listed_price'] = pd.to_numeric(df1['listed_price'], errors='coerce')

In [303]:
# Check for any remaining missing values in the 'listed_price' column after filling and conversion
num_missing_listed = df1['listed_price'].isna().sum()
print(f"Number of missing values in 'listed_price' after cleaning: {num_missing_listed}")


Number of missing values in 'listed_price' after cleaning: 2062


In [304]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   title                     42675 non-null  str    
 1   rating                    41651 non-null  float64
 2   number_of_reviews         41651 non-null  float64
 3   bought_in_last_month      32164 non-null  Int64  
 4   current/discounted_price  40613 non-null  str    
 5   price_on_variant          20071 non-null  object 
 6   listed_price              40613 non-null  float64
 7   is_best_seller            42675 non-null  str    
 8   is_sponsored              42675 non-null  str    
 9   is_couponed               42675 non-null  str    
 10  buy_box_availability      28022 non-null  str    
 11  delivery_details          30955 non-null  str    
 12  sustainability_badges     3408 non-null   str    
 13  image_url                 42675 non-null  str    
 14  product_url      

In [305]:
df1.head(5)

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375.0,300,89.68,NaN,159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457.0,6000,9.99,NaN,15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044.0,2000,314.0,NaN,349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6,35882.0,10000,162.24,$162.24,162.24,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,/Apple-Cancellation-Transparency-Personalized-...,2025-08-21 11:14:29
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8,28988.0,10000,72.74,$72.74,72.74,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...,2025-08-21 11:14:29


<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Quy trình làm sạch cột 'delivery_details'
</h3>

In [306]:
df1['delivery_details']

0                                      Delivery Mon, Sep 1
1                                     Delivery Fri, Aug 29
2                                      Delivery Mon, Sep 1
3                                                      NaN
4                                                      NaN
                               ...                        
42670    FREE delivery Thu, Sep 4Or fastest delivery Tu...
42671    FREE delivery Thu, Sep 4Or fastest delivery Mo...
42672    FREE delivery Thu, Sep 4Or fastest delivery We...
42673    FREE delivery Thu, Sep 4 on $35 of items shipp...
42674    FREE delivery Thu, Sep 4Or fastest delivery We...
Name: delivery_details, Length: 42675, dtype: str

In [307]:
# Display the unique counts of values in 'delivery_details' to inspect inconsistencies
unique_delivery_counts = df1['delivery_details'].value_counts().unique()
print(unique_delivery_counts)


[6189 4364 3700 3278 2164 1238 1012 1003  771  578  545  501  460  413
  357  319  253  192  185  150  129  111  109   98   94   93   92   90
   85   83   80   74   73   68   67   63   62   60   57   55   54   48
   46   40   39   38   37   36   34   33   32   30   29   27   24   22
   21   20   19   16   15   14   13   12   11   10    9    8    7    6
    5    4    3    2    1]


In [308]:
# Extract the date part from 'delivery_details' using regex (e.g., "Sep 12") and ignore day names

df1['delivery_details'] = df1['delivery_details'].str.extract(r'(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)?,?\s*(\w+\s+\d{1,2})')

In [309]:
# Convert 'delivery_details' to datetime, appending a default year (2025) to the extracted month-day

df1['delivery_details'] = pd.to_datetime(df1['delivery_details'] + ' 2025', errors='coerce')

In [310]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   title                     42675 non-null  str           
 1   rating                    41651 non-null  float64       
 2   number_of_reviews         41651 non-null  float64       
 3   bought_in_last_month      32164 non-null  Int64         
 4   current/discounted_price  40613 non-null  str           
 5   price_on_variant          20071 non-null  object        
 6   listed_price              40613 non-null  float64       
 7   is_best_seller            42675 non-null  str           
 8   is_sponsored              42675 non-null  str           
 9   is_couponed               42675 non-null  str           
 10  buy_box_availability      28022 non-null  str           
 11  delivery_details          30692 non-null  datetime64[us]
 12  sustainability_badges     340

<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Quá trình hoàn thiện 'product_url'
</h3>

In [311]:
df1['product_url']

0        /sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...
1        /sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...
2        /sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...
3        /Apple-Cancellation-Transparency-Personalized-...
4        /Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...
                               ...                        
42670    /Elgato-4K-Pro-Internal-Capture/dp/B0DLR3WQWR/...
42671    /Arlo-Essential-Spotlight-Camera-Surveillance/...
42672    /GIGABYTE-FO32U2-32-3840x2160-240Hz-FreeSync-A...
42673    /Monoprice-XLR-Male-4-Inch-Cable/dp/B001UJEKZ6...
42674    /Lorex-8-Channel-Security-Outdoor-Cameras/dp/B...
Name: product_url, Length: 42675, dtype: str

In [312]:
# Cross-verify the first entry of 'product_url' and 'image_url' columns
first_product_url = df1['product_url'].iloc[0]
first_image_url = df1['image_url'].iloc[0]

print(f"First product URL: https://www.amazon.com{first_product_url}")
print(f"First image URL: {first_image_url}")


First product URL: https://www.amazon.com/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxNDQ2OjE3NTU4MDAwNjg6c3BfYXRmX2Jyb3dzZTozMDA2NzE0NTMwMTcyMDI6OjA6Og&url=%2FBOYA-BOYALINK-Microphone-Micophone-Cancelling%2Fdp%2FB0DNZB7TQG%2Fref%3Dsr_1_1_sspa%3Fdib%3DeyJ2IjoiMSJ9.avmZlHCQuVOwikquBqYSIjN8SVcyxtkXHQMPt7Zjzkf4TeZzrZfQETMhdWuWgtTrVz8ITKpXLHvZj0fZRjxqgMPYNMitRqeUoeIwdYfc5nnzJ8m0T8HYeedlh3YSOhJQjeHskevMUQWyg6TtoB2tHcHt-edYPsQ6VwFQTI6avsPrgVpFKrto3ff9TDR9BcRyPwM6AiYn-vh7wA5PP9DjZddhCPf7bVPcHMZ6Hwd40dQDWm9_M8R-LcwKY8wnuWRUSplhfYJBvTjAgsb3Y3y88VGpfqY3V8Rd2ge-woBzUMA.yqwi-krmElGgXIMa8kKUPx1XmaXd9lHaAkMkVVpStxo%26dib_tag%3Dse%26qid%3D1755800068%26refinements%3Dp_n_g-101014971069111%253A119653281011%26s%3Delectronics%26sr%3D1-1-spons%26sp_csd%3Dd2lkZ2V0TmFtZT1zcF9hdGZfYnJvd3Nl%26psc%3D1
First image URL: https://m.media-amazon.com/images/I/71pAqiVEs3L._AC_UL320_.jpg


In [313]:
df1['product_url'].isna().sum()

np.int64(2069)

In [314]:
# Define the Amazon base URL
amazon_base_url = "https://www.amazon.com"

# Complete product URLs safely, handling NaN values
df1['product_url'] = df1['product_url'].apply(
    lambda x: amazon_base_url + x
    if pd.notna(x) and not str(x).startswith(("http://", "https://"))
    else x
)



In [315]:
df1['product_url'].isna().sum()

np.int64(2069)

In [316]:
df1['product_url']

0        https://www.amazon.com/sspa/click?ie=UTF8&spc=...
1        https://www.amazon.com/sspa/click?ie=UTF8&spc=...
2        https://www.amazon.com/sspa/click?ie=UTF8&spc=...
3        https://www.amazon.com/Apple-Cancellation-Tran...
4        https://www.amazon.com/Apple-MX542LL-A-AirTag-...
                               ...                        
42670    https://www.amazon.com/Elgato-4K-Pro-Internal-...
42671    https://www.amazon.com/Arlo-Essential-Spotligh...
42672    https://www.amazon.com/GIGABYTE-FO32U2-32-3840...
42673    https://www.amazon.com/Monoprice-XLR-Male-4-In...
42674    https://www.amazon.com/Lorex-8-Channel-Securit...
Name: product_url, Length: 42675, dtype: str

<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Làm sạch/Định dạng cột 'collected_at'
</h3>

In [317]:
df1['collected_at']

0        2025-08-21 11:14:29
1        2025-08-21 11:14:29
2        2025-08-21 11:14:29
3        2025-08-21 11:14:29
4        2025-08-21 11:14:29
                ...         
42670    2025-08-30 19:56:33
42671    2025-08-30 19:56:33
42672    2025-08-30 19:56:33
42673    2025-08-30 19:56:33
42674    2025-08-30 19:56:33
Name: collected_at, Length: 42675, dtype: str

In [318]:
# Convert 'collected_at' column to datetime format for consistency and analysis

df1['collected_at'] = pd.to_datetime(df1['collected_at'])

In [319]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   title                     42675 non-null  str           
 1   rating                    41651 non-null  float64       
 2   number_of_reviews         41651 non-null  float64       
 3   bought_in_last_month      32164 non-null  Int64         
 4   current/discounted_price  40613 non-null  str           
 5   price_on_variant          20071 non-null  object        
 6   listed_price              40613 non-null  float64       
 7   is_best_seller            42675 non-null  str           
 8   is_sponsored              42675 non-null  str           
 9   is_couponed               42675 non-null  str           
 10  buy_box_availability      28022 non-null  str           
 11  delivery_details          30692 non-null  datetime64[us]
 12  sustainability_badges     340

<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Đánh giá cột tiêu đề và tạo cột mới "Category" với ánh xạ từ khóa
</h3>

In [320]:
import re

# Clean text function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

# ✅ Expanded category keywords based on your file analysis
category_keywords = {
    'Laptops': [
        'laptop', 'notebook', 'macbook', 'chromebook', 'ultrabook', 'acer', 'asus', 'dell', 'lenovo', 'hp', 'core',
        'intel', 'ryzen', 'surface', 'thinkpad', 'ideapad'
    ],
    'Phones': [
        'phone', 'iphone', 'smartphone', 'samsung', 'android', 'galaxy', 'pixel', 'oneplus', 'xiaomi', 'oppo',
        'realme', 'huawei', 'vivo', 'nokia', 'motorola'
    ],
    'Headphones': [
        'headphone', 'headset', 'earphone', 'earbuds', 'airpods', 'beats', 'sony wh', 'wireless buds', 'neckband'
    ],
    'Chargers & Cables': [
        'charger', 'charging', 'cable', 'adapter', 'dock', 'usb c', 'type c', 'lightning', 'power adapter', 'usb cable'
    ],
    'Cameras': [
        'camera', 'dslr', 'mirrorless', 'canon', 'nikon', 'gopro', 'instax', 'webcam', 'camcorder', 'security camera'
    ],
    'Storage': [
        'ssd', 'hard drive', 'memory card', 'flash drive', 'pendrive', 'hdd', 'storage', 'micro sd', 'sd card'
    ],
    'Smart Home': [
        'alexa', 'echo', 'smart plug', 'smart bulb', 'smart home', 'nest', 'homekit', 'smart switch'
    ],
    'TV & Display': [
        'monitor', 'display', 'tv', 'screen', 'projector', 'oled', 'led', 'curved monitor', 'uhd', '4k'
    ],
    'Power & Batteries': [
        'battery', 'power bank', 'rechargeable', 'aa', 'aaa', 'portable power', 'cell'
    ],
    'Networking': [
        'wifi', 'router', 'modem', 'ethernet', 'access point', 'mesh', 'network switch'
    ],
    'Wearables': [
        'smartwatch', 'fitness band', 'fitbit', 'watch', 'garmin', 'amazfit'
    ],
    'Speakers': [
        'speaker', 'soundbar', 'subwoofer', 'bluetooth speaker', 'party speaker', 'home theater'
    ],
    'Printers & Scanners': [
        'printer', 'scanner', 'inkjet', 'laserjet', 'photocopier', 'all in one printer'
    ],
    'Gaming': [
        'gaming console', 'playstation', 'ps5', 'ps4', 'xbox', 'nintendo', 'joystick', 'controller', 'gaming mouse',
        'gaming keyboard', 'gaming chair'
    ],
    'Other Electronics': []
}

# ✅ Simple mapping function (No Scoring)
def assign_category_simple(title):
    title_clean = clean_text(title)
    for category, keywords in category_keywords.items():
        for kw in keywords:
            if kw in title_clean:
                return category
    return 'Other Electronics'

# ✅ Apply to dataframe

df1['category'] = df1['title'].apply(assign_category_simple)

# ✅ Show category distribution
print(df1['category'].value_counts())

category
Other Electronics      8755
Laptops                8693
Phones                 6563
Cameras                3677
Power & Batteries      2877
TV & Display           2630
Chargers & Cables      1833
Storage                1630
Speakers               1345
Networking             1070
Headphones              997
Printers & Scanners     877
Gaming                  809
Smart Home              465
Wearables               454
Name: count, dtype: int64


In [321]:
# Display a random sample of 10 rows for 'title' and 'category' columns to verify category assignment
df1.loc[:, ['title', 'category']].sample(10)


,title,category
8059,MEM 2 * 16G|GSK F5-6000J3636F16GX2-FX5,Other Electronics
30932,Dell Keyboard,Laptops
39311,Mounting Dream Sliding TV Wall Mount for 42-86...,TV & Display
5177,RØDE Lavalier II Premium Ultra-Low-Profile Lav...,Phones
14028,"NOCO NUSB211NA 10W USB Power Adapter, 2.1A 5V ...",Phones
1792,RØDE NT1 Signature Series Condenser Microphone...,Phones
33372,Lenovo ThinkPad P16 Gen 2 Intel Core i7-14700H...,Laptops
21111,"Avery Printable Greeting Cards with Envelopes,...",Printers & Scanners
38444,"Skytech Gaming Azure 3 Desktop PC, Ryzen 7 780...",Laptops
19147,Pacsafe Vibe 325 Anti Theft Crossbody Casual D...,Other Electronics


<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Tạo cột 'discount_percentage'
</h3>

In [323]:
# 1. Clean and convert 'current/discounted_price' to float first
df1['current/discounted_price'] = (
    df1['current/discounted_price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df1['current/discounted_price'] = pd.to_numeric(
    df1['current/discounted_price'], errors='coerce'
)

# 2. Create 'discount_percentage' column based on listed_price and current/discounted_price
df1['discount_percentage'] = (
    (df1['listed_price'] - df1['current/discounted_price'])
    / df1['listed_price']
) * 100

# 3. Round to 2 decimal places for better readability
df1['discount_percentage'] = df1['discount_percentage'].round(2)

# 4. Verify the first few entries
df1[['listed_price', 'current/discounted_price', 'discount_percentage']].head(
    10
)

,listed_price,current/discounted_price,discount_percentage
0,159.00,89.68,43.60
1,15.99,9.99,37.52
2,349.00,314.00,10.03
3,162.24,162.24,0.00
4,72.74,72.74,0.00
5,99.95,99.95,0.00
6,88.11,88.11,0.00
7,23.04,23.04,0.00
8,16.99,16.99,0.00
9,284.05,284.05,0.00


<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Xóa cột không cần thiết "price_on_variant" tại thời điểm này
</h3>

In [324]:
# Drop the 'price_on_variant' column as it's no longer needed
df1.drop(columns=['price_on_variant'], inplace=True)

# Verify the remaining columns
print(df1.columns)


Index(['title', 'rating', 'number_of_reviews', 'bought_in_last_month',
       'current/discounted_price', 'listed_price', 'is_best_seller',
       'is_sponsored', 'is_couponed', 'buy_box_availability',
       'delivery_details', 'sustainability_badges', 'image_url', 'product_url',
       'collected_at', 'category', 'discount_percentage'],
      dtype='str')


<h3 style="
  font-size:22px;
  font-weight:700;
  background:linear-gradient(90deg,#2563eb,#9333ea);
  color:#fff;
  padding:10px 16px;
  border-radius:8px;
  border:2px solid #ffffff33;
  display:inline-block;
  margin:14px 0;
">
  Đổi tên cột để rõ ràng và nhất quán
</h3>

In [325]:
# Rename columns for clarity and consistency
df1.rename(columns={
    'title': 'product_title',
    'rating': 'product_rating',
    'number_of_reviews': 'total_reviews',
    'bought_in_last_month': 'purchased_last_month',
    'current/discounted_price': 'discounted_price',
    'listed_price': 'original_price',
    'is_couponed': 'has_coupon',
    'delivery_details': 'delivery_date',
    'sustainability_badges': 'sustainability_tags',
    'image_url': 'product_image_url',
    'product_url': 'product_page_url',
    'collected_at': 'data_collected_at',
    'category': 'product_category',
    'discount_percentage': 'discount_percentage'
}, inplace=True)

# Verify column names after renaming
print(df1.columns)


Index(['product_title', 'product_rating', 'total_reviews',
       'purchased_last_month', 'discounted_price', 'original_price',
       'is_best_seller', 'is_sponsored', 'has_coupon', 'buy_box_availability',
       'delivery_date', 'sustainability_tags', 'product_image_url',
       'product_page_url', 'data_collected_at', 'product_category',
       'discount_percentage'],
      dtype='str')


In [326]:
df1

,product_title,product_rating,total_reviews,purchased_last_month,discounted_price,original_price,is_best_seller,is_sponsored,has_coupon,buy_box_availability,delivery_date,sustainability_tags,product_image_url,product_page_url,data_collected_at,product_category,discount_percentage
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375.0,300,89.68,159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,2025-09-01,Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Phones,43.60
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457.0,6000,9.99,15.99,No Badge,Sponsored,No Coupon,Add to cart,2025-08-29,NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Laptops,37.52
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044.0,2000,314.00,349.00,No Badge,Sponsored,No Coupon,Add to cart,2025-09-01,NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Laptops,10.03
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6,35882.0,10000,162.24,162.24,Best Seller,Organic,No Coupon,NaN,NaT,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,https://www.amazon.com/Apple-Cancellation-Tran...,2025-08-21 11:14:29,Phones,0.00
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8,28988.0,10000,72.74,72.74,No Badge,Organic,No Coupon,NaN,NaT,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,https://www.amazon.com/Apple-MX542LL-A-AirTag-...,2025-08-21 11:14:29,Phones,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42670,"Elgato 4K Pro, Internal Capture Card: 8K60 Pas...",5.0,1.0,100,195.99,195.99,No Badge,Organic,No Coupon,NaN,2025-09-04,NaN,https://m.media-amazon.com/images/I/51KCB+egEs...,https://www.amazon.com/Elgato-4K-Pro-Internal-...,2025-08-30 19:56:33,TV & Display,0.00
42671,"Arlo Essential Spotlight Camera, Wireless Secu...",4.2,20.0,200,89.99,89.99,No Badge,Organic,Save $25.00 with coupon,Add to cart,2025-09-04,NaN,https://m.media-amazon.com/images/I/51jV+o1LZE...,https://www.amazon.com/Arlo-Essential-Spotligh...,2025-08-30 19:56:33,Cameras,0.00
42672,"GIGABYTE - AORUS FO32U2-32"" QD OLED Gaming Mon...",4.3,57.0,50,899.99,1099.99,Save 18%,Organic,No Coupon,Add to cart,2025-09-04,NaN,https://m.media-amazon.com/images/I/71ySPkNLkG...,https://www.amazon.com/GIGABYTE-FO32U2-32-3840...,2025-08-30 19:56:33,Chargers & Cables,18.18
42673,Monoprice XLR Male to 1/4-Inch TRS Male Cable ...,4.7,7102.0,500,10.39,15.98,No Badge,Organic,No Coupon,Add to cart,2025-09-04,NaN,https://m.media-amazon.com/images/I/411c0JFJ79...,https://www.amazon.com/Monoprice-XLR-Male-4-In...,2025-08-30 19:56:33,Chargers & Cables,34.98


In [327]:
df1.shape

(42675, 17)

In [328]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   product_title         42675 non-null  str           
 1   product_rating        41651 non-null  float64       
 2   total_reviews         41651 non-null  float64       
 3   purchased_last_month  32164 non-null  Int64         
 4   discounted_price      40613 non-null  float64       
 5   original_price        40613 non-null  float64       
 6   is_best_seller        42675 non-null  str           
 7   is_sponsored          42675 non-null  str           
 8   has_coupon            42675 non-null  str           
 9   buy_box_availability  28022 non-null  str           
 10  delivery_date         30692 non-null  datetime64[us]
 11  sustainability_tags   3408 non-null   str           
 12  product_image_url     42675 non-null  str           
 13  product_page_url      40606

In [331]:
# Lưu file sang folder data
df1.to_csv('../data/amazon_electronics_data_cleaned.csv', index=False)